# 🚀 NASA CMAPSS Veri Seti - Model Karşılaştırma Analizi

Bu notebook'ta 4 farklı CMAPSS dataseti üzerinde çeşitli makine öğrenmesi ve deep learning modellerini karşılaştırıyoruz.

## Datasets:
- **FD001**: Tek operasyon modu, tek arıza tipi
- **FD002**: 6 operasyon modu, tek arıza tipi  
- **FD003**: Tek operasyon modu, 2 arıza tipi
- **FD004**: 6 operasyon modu, 2 arıza tipi

## Modeller:
1. **XGBoost** (Baseline - mevcut)
2. **Random Forest**
3. **SVR (Support Vector Regression)**
4. **Linear Regression**
5. **LSTM (Deep Learning)**
6. **GRU (Deep Learning)**
7. **CNN-LSTM Hybrid (Deep Learning)**

## 1. Kütüphaneleri Yükle

In [4]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['OMP_NUM_THREADS'] = '1'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression

# XGBoost
from xgboost import XGBRegressor

import tensorflow as tf

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Conv1D, MaxPooling1D, Flatten, Input, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Joblib
import joblib

# Zaman takibi
import time

print("✅ Kütüphaneler yüklendi")
print(f"TensorFlow version: {tf.__version__}")

## 2. Veri Yükleme ve Ön İşleme Fonksiyonları

In [ ]:
# Sütun isimleri
sensor_columns = ['sensor_' + str(i) for i in range(1, 22)]
setting_columns = ['setting_' + str(i) for i in range(1, 4)]
column_names = ['unit_number', 'time_in_cycles'] + setting_columns + sensor_columns

def load_cmapss_data(dataset_name='FD001', data_dir='CMaps'):
    """
    CMAPSS veri setini yükler ve RUL hesaplar
    
    Parameters:
    - dataset_name: 'FD001', 'FD002', 'FD003', 'FD004'
    - data_dir: Veri klasörü yolu
    
    Returns:
    - train_df: Eğitim dataframe (RUL ile)
    - test_df: Test dataframe
    - rul_df: Test RUL değerleri
    """
    # Train verisi
    train_path = f'{data_dir}/train_{dataset_name}.txt'
    train_df = pd.read_csv(train_path, sep='\s+', header=None, names=column_names)
    
    # Test verisi
    test_path = f'{data_dir}/test_{dataset_name}.txt'
    test_df = pd.read_csv(test_path, sep='\s+', header=None, names=column_names)
    
    # RUL değerleri
    rul_path = f'{data_dir}/RUL_{dataset_name}.txt'
    rul_df = pd.read_csv(rul_path, sep='\s+', header=None, names=['RUL'])
    
    # Train için RUL hesapla
    train_df['RUL'] = train_df.groupby('unit_number')['time_in_cycles'].transform(max) - train_df['time_in_cycles']
    
    # Test için RUL ekle (son cycle'a göre)
    test_df['max_cycle'] = test_df.groupby('unit_number')['time_in_cycles'].transform(max)
    test_last = test_df.groupby('unit_number').last().reset_index()
    test_last['RUL'] = rul_df['RUL'].values
    
    return train_df, test_df, test_last, rul_df

print("✅ Veri yükleme fonksiyonları hazır")

## 3. Veri Setlerini Yükle

In [ ]:
datasets = {}

for ds_name in ['FD001', 'FD002', 'FD003', 'FD004']:
    print(f"\n📂 {ds_name} yükleniyor...")
    train_df, test_df, test_last, rul_df = load_cmapss_data(ds_name)
    
    datasets[ds_name] = {
        'train': train_df,
        'test': test_df,
        'test_last': test_last,
        'rul': rul_df
    }
    
    print(f"   Train: {len(train_df)} satır, {train_df['unit_number'].nunique()} motor")
    print(f"   Test: {len(test_df)} satır, {test_df['unit_number'].nunique()} motor")
    print(f"   RUL range: {train_df['RUL'].min():.0f} - {train_df['RUL'].max():.0f}")

print("\n✅ Tüm veri setleri yüklendi!")

## 4. Feature Engineering

In [ ]:
def prepare_features(df, feature_cols=None):
    """Özellikleri hazırlar (sabit değerleri kaldır)"""
    if feature_cols is None:
        feature_cols = [col for col in df.columns if 'sensor' in col or 'setting' in col]
    
    # Sabit değerleri kaldır (variance = 0)
    variance = df[feature_cols].var()
    feature_cols = variance[variance > 0.01].index.tolist()
    
    return feature_cols

# Her dataset için feature'ları belirle
for ds_name in datasets.keys():
    train_df = datasets[ds_name]['train']
    feature_cols = prepare_features(train_df)
    datasets[ds_name]['features'] = feature_cols
    print(f"{ds_name}: {len(feature_cols)} özellik seçildi")

print("\n✅ Feature engineering tamamlandı")

## 5. ML Model Tanımları

In [ ]:
def get_ml_models():
    """Geleneksel makine öğrenmesi modellerini döndürür"""
    models = {
        'Linear Regression': LinearRegression(),
        
        'Random Forest': RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_split=10,
            random_state=42,
            n_jobs=-1
        ),
        
        'XGBoost': XGBRegressor(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        ),
        
        'SVR': SVR(
            kernel='rbf',
            C=100,
            gamma='scale'
        )
    }
    
    return models

print("✅ ML modelleri tanımlandı")

## 6. Deep Learning Model Tanımları

In [ ]:
def create_lstm_model(input_shape, units=64):
    """LSTM modeli oluşturur"""
    model = Sequential([
        LSTM(units, activation='tanh', return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(units//2, activation='tanh'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

def create_gru_model(input_shape, units=64):
    """GRU modeli oluşturur"""
    model = Sequential([
        GRU(units, activation='tanh', return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        GRU(units//2, activation='tanh'),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

def create_cnn_lstm_model(input_shape, filters=64, units=50):
    """CNN-LSTM hybrid model oluşturur"""
    model = Sequential([
        Conv1D(filters=filters, kernel_size=3, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),
        
        Conv1D(filters=filters//2, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),
        
        LSTM(units, activation='tanh'),
        Dropout(0.2),
        
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

def create_sequences(df, feature_cols, sequence_length=30):
    """Zaman serisi sequence'ları oluşturur (Deep Learning için)"""
    X_seq = []
    y_seq = []
    
    for unit in df['unit_number'].unique():
        unit_data = df[df['unit_number'] == unit].sort_values('time_in_cycles')
        X_unit = unit_data[feature_cols].values
        y_unit = unit_data['RUL'].values
        
        for i in range(sequence_length, len(X_unit)):
            X_seq.append(X_unit[i-sequence_length:i])
            y_seq.append(y_unit[i])
    
    return np.array(X_seq), np.array(y_seq)

print("✅ Deep Learning modelleri tanımlandı")

## 6.1 Evolved Modeller (Attention + Bidirectional)


In [ ]:
class Attention(tf.keras.layers.Layer):
    def __init__(self, **kwargs): super(Attention, self).__init__(**kwargs)
    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1), initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1), initializer="zeros")
        super(Attention, self).build(input_shape)
    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return tf.keras.backend.sum(output, axis=1)

def create_evolved_lstm(input_shape):
    inputs = Input(shape=input_shape)
    x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
    x = Dropout(0.2)(x)
    x = Bidirectional(LSTM(32, return_sequences=True))(x)
    x = Attention()(x)
    x = Dense(32, activation="relu")(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

def create_evolved_gru(input_shape):
    inputs = Input(shape=input_shape)
    x = Bidirectional(GRU(64, return_sequences=True))(inputs)
    x = Dropout(0.2)(x)
    x = Bidirectional(GRU(32, return_sequences=True))(x)
    x = Attention()(x)
    x = Dense(32, activation="relu")(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

print("✅ Evrilmiş (Attention) modeller tanımlandı")

## 7. Model Eğitim ve Değerlendirme - ML Modelleri

In [ ]:
# Sonuçları saklamak için liste
all_results = []

# Parametreler
TEST_SIZE = 0.2
RANDOM_STATE = 42

print("🚀 ML Modelleri Eğitiliyor...\n")
print("="*80)

for ds_name in ['FD001', 'FD002', 'FD003', 'FD004']:
    print(f"\n📊 Dataset: {ds_name}")
    print("-" * 80)
    
    # Veriyi hazırla
    train_df = datasets[ds_name]['train']
    feature_cols = datasets[ds_name]['features']
    
    X = train_df[feature_cols].values
    y = train_df['RUL'].values
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    
    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # ML modellerini eğit
    ml_models = get_ml_models()
    
    for model_name, model in ml_models.items():
        print(f"   🔧 {model_name} eğitiliyor...")
        
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        train_time = time.time() - start_time
        
        # Tahmin
        y_pred = model.predict(X_test_scaled)
        
        # Değerlendirme
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        # MAPE (Mean Absolute Percentage Error)
        mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100
        
        # Threshold-based Accuracy (±10 cycle tolerans)
        threshold = 10
        accuracy_10 = np.mean(np.abs(y_test - y_pred) <= threshold) * 100
        
        # Threshold-based Accuracy (±5 cycle tolerans)
        accuracy_5 = np.mean(np.abs(y_test - y_pred) <= 5) * 100
        
        results = {
            'Model': model_name,
            'Dataset': ds_name,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
            'MAPE': mape,
            'Accuracy_5': accuracy_5,
            'Accuracy_10': accuracy_10,
            'Train_Time': train_time
        }
        all_results.append(results)
        
        print(f"      ✅ MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.3f} | MAPE: {mape:.2f}% | Acc(±5): {accuracy_5:.1f}% | Acc(±10): {accuracy_10:.1f}% | Time: {train_time:.2f}s")

print("\n" + "="*80)
print("✅ ML modelleri tamamlandı!")

## 8. Deep Learning Modelleri Eğitimi

In [ ]:
# Deep Learning parametreleri
SEQUENCE_LENGTH = 30
EPOCHS = 50
BATCH_SIZE = 256
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Callbacks - daha akıllı ayarlar
early_stop = EarlyStopping(
    monitor="val_loss", 
    patience=15,  # Daha fazla sabır
    restore_best_weights=True,
    min_delta=0.001  # Minimum iyileşme
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", 
    factor=0.5, 
    patience=8,  # Daha fazla sabır
    min_lr=1e-7
)

print("🧠 Deep Learning Modelleri Eğitiliyor...")
print("="*80)

for ds_name in ["FD001", "FD002", "FD003", "FD004"]:
    print(f"\n🧠 Deep Learning - Dataset: {ds_name}")
    print("-" * 80)
    
    # Veriyi hazırla
    train_df = datasets[ds_name]["train"]
    feature_cols = datasets[ds_name]["features"]
    
    # Sequence oluştur
    print("   📦 Sequencelar oluşturuluyor...")
    X_seq, y_seq = create_sequences(train_df, feature_cols, SEQUENCE_LENGTH)
    print(f"      ✅ {len(X_seq)} sequence oluşturuldu")
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_seq, y_seq, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    
    # Scaling
    scaler = StandardScaler()
    # Reshape for scaling (samples, time_steps, features) -> (samples*time_steps, features)
    X_train_reshape = X_train.reshape(-1, X_train.shape[-1])
    X_train_scaled = scaler.fit_transform(X_train_reshape)
    X_train_scaled = X_train_scaled.reshape(X_train.shape)
    
    X_test_reshape = X_test.reshape(-1, X_test.shape[-1])
    X_test_scaled = scaler.transform(X_test_reshape)
    X_test_scaled = X_test_scaled.reshape(X_test.shape)
    
    # Input shape definition
    input_shape = (X_train_scaled.shape[1], X_train_scaled.shape[2])
    print(f"      ✅ Input shape: {input_shape}")
    
    # Deep Learning modelleri
    dl_models = {
        "LSTM": create_lstm_model(input_shape),
        "GRU": create_gru_model(input_shape),
        "CNN-LSTM": create_cnn_lstm_model(input_shape),
        "Evolved-LSTM": create_evolved_lstm(input_shape),
        "Evolved-GRU": create_evolved_gru(input_shape)
    }
    
    for model_name, model in dl_models.items():
        print(f"\n   🔧 {model_name} eğitiliyor...")
        sys.stdout.flush()
        
        try:
            start_time = time.time()
            
            # Model eğitimi - verbose=1 ile ilerleme göster
            history = model.fit(
                X_train_scaled, y_train,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                validation_split=0.2,
                callbacks=[early_stop, reduce_lr],
                verbose=1  # İlerleme göster
            )
            
            train_time = time.time() - start_time
            
            # Epoch kontrolü
            epochs_trained = len(history.history["loss"])
            if epochs_trained == 0:
                print(f"      ❌ HATA: {model_name} hiç epoch çalıştırmadı!")
                continue
            
            if train_time < 1.0:
                print(f"      ⚠️  UYARI: {model_name} eğitim süresi çok kısa ({train_time:.3f}s) - kontrol edin!")
            
            # Tahmin
            y_pred = model.predict(X_test_scaled, verbose=0).flatten()
            
            # Değerlendirme
            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)
            
            # MAPE
            mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100
            
            # Accuracy
            accuracy_10 = np.mean(np.abs(y_test - y_pred) <= 10) * 100
            accuracy_5 = np.mean(np.abs(y_test - y_pred) <= 5) * 100
            
            all_results.append({
                "Model": model_name,
                "Dataset": ds_name,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2,
                "MAPE": mape,
                "Accuracy_5": accuracy_5,
                "Accuracy_10": accuracy_10,
                "Train_Time": train_time,
                "Epochs": epochs_trained
            })
            
            print(f"      ✅ MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.3f} | MAPE: {mape:.2f}% | Acc(±5): {accuracy_5:.1f}% | Epochs: {epochs_trained} | Time: {train_time:.2f}s")
            sys.stdout.flush()
            
        except Exception as e:
            print(f"      ❌ HATA: {model_name} eğitilirken hata: {type(e).__name__}: {e}")
            import traceback
            traceback.print_exc()
            continue

print("\n✅ Tüm Deep Learning eğitimleri tamamlandı!")


## 9. Sonuçları Görselleştir

In [ ]:
# Sonuçları DataFrame'e çevir
results_df = pd.DataFrame(all_results)
results_df = results_df.round(3)

print("\n📊 TÜM MODEL SONUÇLARI")
print("="*100)
print(results_df.to_string(index=False))
print("="*100)

# Sonuçları kaydet
results_df.to_csv('model_comparison_results.csv', index=False)
print("\n💾 Sonuçlar 'model_comparison_results.csv' olarak kaydedildi")

### 9.1 Dataset Bazında MAE Karşılaştırması

In [ ]:

# ML ve DL modellerini ayır
ml_models_list = ['Linear Regression', 'Random Forest', 'XGBoost', 'SVR']
dl_models_list = ['LSTM', 'GRU', 'CNN-LSTM', 'Evolved-LSTM', 'Evolved-GRU']

fig, axes = plt.subplots(4, 2, figsize=(18, 20))
fig.suptitle('Dataset Bazında Model Performansı (MAE) - ML vs DL', fontsize=16, fontweight='bold')

for idx, ds_name in enumerate(['FD001', 'FD002', 'FD003', 'FD004']):
    # ML Modelleri (sol sütun)
    ml_data = results_df[(results_df['Dataset'] == ds_name) & (results_df['Model'].isin(ml_models_list))].sort_values('MAE')
    ax_ml = axes[idx, 0]
    colors_ml = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(ml_data))]
    ax_ml.barh(ml_data['Model'], ml_data['MAE'], color=colors_ml, alpha=0.8)
    ax_ml.set_xlabel('MAE (Lower is Better)', fontsize=12)
    ax_ml.set_title(f'{ds_name} - ML Modelleri', fontsize=14, fontweight='bold')
    ax_ml.grid(axis='x', alpha=0.3)
    if len(ml_data) > 0:
        best_mae_ml = ml_data.iloc[0]['MAE']
        ax_ml.text(best_mae_ml, 0, f' ⭐ {best_mae_ml:.2f}', va='center', fontweight='bold')
    
    # DL Modelleri (sağ sütun)
    dl_data = results_df[(results_df['Dataset'] == ds_name) & (results_df['Model'].isin(dl_models_list))].sort_values('MAE')
    ax_dl = axes[idx, 1]
    colors_dl = ['#e74c3c' if i == 0 else '#f39c12' for i in range(len(dl_data))]
    ax_dl.barh(dl_data['Model'], dl_data['MAE'], color=colors_dl, alpha=0.8)
    ax_dl.set_xlabel('MAE (Lower is Better)', fontsize=12)
    ax_dl.set_title(f'{ds_name} - Deep Learning Modelleri', fontsize=14, fontweight='bold')
    ax_dl.grid(axis='x', alpha=0.3)
    if len(dl_data) > 0:
        best_mae_dl = dl_data.iloc[0]['MAE']
        ax_dl.text(best_mae_dl, 0, f' ⭐ {best_mae_dl:.2f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison_mae_separated.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Grafik 'model_comparison_mae_separated.png' olarak kaydedildi")


### 9.2 Model Bazında Ortalama Performans

In [ ]:

# ML ve DL modellerini ayır
ml_models_list = ['Linear Regression', 'Random Forest', 'XGBoost', 'SVR']
dl_models_list = ['LSTM', 'GRU', 'CNN-LSTM', 'Evolved-LSTM', 'Evolved-GRU']

# ML modelleri için ortalama
ml_results = results_df[results_df['Model'].isin(ml_models_list)]
ml_avg = ml_results.groupby('Model')[['MAE', 'RMSE', 'R2']].mean().sort_values('MAE')

# DL modelleri için ortalama
dl_results = results_df[results_df['Model'].isin(dl_models_list)]
dl_avg = dl_results.groupby('Model')[['MAE', 'RMSE', 'R2']].mean().sort_values('MAE')

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Model Bazında Ortalama Performans - ML vs DL (Tüm Datasetler)', fontsize=16, fontweight='bold')

# ML Modelleri (üst satır)
axes[0, 0].barh(ml_avg.index, ml_avg['MAE'], color='skyblue', alpha=0.8)
axes[0, 0].set_xlabel('MAE')
axes[0, 0].set_title('ML - Mean Absolute Error')
axes[0, 0].grid(axis='x', alpha=0.3)

axes[0, 1].barh(ml_avg.index, ml_avg['RMSE'], color='lightcoral', alpha=0.8)
axes[0, 1].set_xlabel('RMSE')
axes[0, 1].set_title('ML - Root Mean Squared Error')
axes[0, 1].grid(axis='x', alpha=0.3)

axes[0, 2].barh(ml_avg.index, ml_avg['R2'], color='lightgreen', alpha=0.8)
axes[0, 2].set_xlabel('R²')
axes[0, 2].set_title('ML - R² Score')
axes[0, 2].grid(axis='x', alpha=0.3)

# DL Modelleri (alt satır)
axes[1, 0].barh(dl_avg.index, dl_avg['MAE'], color='#e74c3c', alpha=0.8)
axes[1, 0].set_xlabel('MAE')
axes[1, 0].set_title('Deep Learning - Mean Absolute Error')
axes[1, 0].grid(axis='x', alpha=0.3)

axes[1, 1].barh(dl_avg.index, dl_avg['RMSE'], color='#f39c12', alpha=0.8)
axes[1, 1].set_xlabel('RMSE')
axes[1, 1].set_title('Deep Learning - Root Mean Squared Error')
axes[1, 1].grid(axis='x', alpha=0.3)

axes[1, 2].barh(dl_avg.index, dl_avg['R2'], color='#27ae60', alpha=0.8)
axes[1, 2].set_xlabel('R²')
axes[1, 2].set_title('Deep Learning - R² Score')
axes[1, 2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison_avg_separated.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Grafik 'model_comparison_avg_separated.png' olarak kaydedildi")


### 9.3 Heatmap - Model vs Dataset

In [ ]:
# Pivot table oluştur
pivot_mae = results_df.pivot(index='Model', columns='Dataset', values='MAE')
pivot_r2 = results_df.pivot(index='Model', columns='Dataset', values='R2')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model vs Dataset Performance Heatmap', fontsize=16, fontweight='bold')

# MAE Heatmap
sns.heatmap(pivot_mae, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0], cbar_kws={'label': 'MAE'})
axes[0].set_title('MAE (Lower is Better)')
axes[0].set_xlabel('Dataset')
axes[0].set_ylabel('Model')

# R2 Heatmap
sns.heatmap(pivot_r2, annot=True, fmt='.3f', cmap='YlGn', ax=axes[1], cbar_kws={'label': 'R²'})
axes[1].set_title('R² Score (Higher is Better)')
axes[1].set_xlabel('Dataset')
axes[1].set_ylabel('Model')

plt.tight_layout()
plt.savefig('model_comparison_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Grafik 'model_comparison_heatmap.png' olarak kaydedildi")

### 9.4 Eğitim Süresi Karşılaştırması

In [ ]:
# Ortalama eğitim süresi
time_avg = results_df.groupby('Model')['Train_Time'].mean().sort_values()

plt.figure(figsize=(12, 6))
bars = plt.barh(time_avg.index, time_avg.values, color='steelblue', alpha=0.8)

# Renklendirme (hızlı: yeşil, yavaş: kırmızı)
colors = ['green' if t < 10 else 'orange' if t < 60 else 'red' for t in time_avg.values]
for bar, color in zip(bars, colors):
    bar.set_color(color)
    bar.set_alpha(0.7)

plt.xlabel('Ortalama Eğitim Süresi (saniye)', fontsize=12)
plt.title('Model Eğitim Süreleri Karşılaştırması', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

# Değerleri ekle
for i, v in enumerate(time_avg.values):
    plt.text(v, i, f' {v:.1f}s', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_training_time.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Grafik 'model_training_time.png' olarak kaydedildi")

## 10. En İyi Modeller Özeti

In [ ]:

print("\n" + "="*100)
print("🏆 EN İYİ MODELLER - ML vs DL")
print("="*100)

ml_models_list = ['Linear Regression', 'Random Forest', 'XGBoost', 'SVR']
dl_models_list = ['LSTM', 'GRU', 'CNN-LSTM', 'Evolved-LSTM', 'Evolved-GRU']

# Her dataset için en iyi ML ve DL modeli
for ds_name in ['FD001', 'FD002', 'FD003', 'FD004']:
    print(f"\n📊 {ds_name}:")
    
    # En iyi ML modeli
    ml_best = results_df[(results_df['Dataset'] == ds_name) & (results_df['Model'].isin(ml_models_list))].sort_values('MAE').iloc[0]
    print(f"   🤖 ML: {ml_best['Model']} | MAE: {ml_best['MAE']:.2f} | R²: {ml_best['R2']:.3f}")
    
    # En iyi DL modeli
    dl_best = results_df[(results_df['Dataset'] == ds_name) & (results_df['Model'].isin(dl_models_list))].sort_values('MAE').iloc[0]
    print(f"   🧠 DL: {dl_best['Model']} | MAE: {dl_best['MAE']:.2f} | R²: {dl_best['R2']:.3f}")

# Genel en iyi ML modeli
ml_overall = results_df[results_df['Model'].isin(ml_models_list)].groupby('Model')['MAE'].mean().sort_values()
ml_best_model = ml_overall.index[0]
ml_best_mae = ml_overall.iloc[0]

# Genel en iyi DL modeli
dl_overall = results_df[results_df['Model'].isin(dl_models_list)].groupby('Model')['MAE'].mean().sort_values()
dl_best_model = dl_overall.index[0]
dl_best_mae = dl_overall.iloc[0]

print(f"\n\n🌟 GENEL EN İYİ ML MODEL: {ml_best_model}")
print(f"   📊 Ortalama MAE: {ml_best_mae:.2f}")

print(f"\n🌟 GENEL EN İYİ DL MODEL: {dl_best_model}")
print(f"   📊 Ortalama MAE: {dl_best_mae:.2f}")

print("\n" + "="*100)
